## Rule-based data integration pipeline (embedding blocker)

In [1]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = CORR_DIR / "pipeline_corr"

In [2]:
from PyDI.io import load_parquet

amazon_books = load_parquet(
    DATA_DIR / "amazon_new.parquet",
    name="amazon_books"
)

goodreads = load_parquet(
    DATA_DIR / "goodreads_new.parquet",
    name="goodreads"
)

metabooks = load_parquet(
  DATA_DIR / "metabooks_new.parquet",
  name="metabooks"
)

In [3]:
from PyDI.entitymatching import EmbeddingBlocker

embedding_blocker_m2a = EmbeddingBlocker(
    metabooks, amazon_books,
    text_cols=['title_norm', 'author_norm'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)


embedding_blocker_m2g = EmbeddingBlocker(
    metabooks, goodreads,
    text_cols=['title_norm', 'author_norm'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

/Users/abd/Developer/team_project_data_integration/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import RuleBasedMatcher


comparators_m2a = [
    StringComparator(column="title_norm", similarity_function="cosine"),
    StringComparator(column="author_norm", similarity_function="cosine", preprocess=str.lower),
    NumericComparator(column="publish_year", max_difference=1),
]

comparators_m2g = comparators_m2a + [
    StringComparator(column="genres", similarity_function="jaccard",
                     preprocess=str.lower, list_strategy="concatenate"),
    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="rating", max_difference=0.2),
]

In [ ]:
matcher = RuleBasedMatcher()

# matching metabooks > amazon
em_corr_m2a = matcher.match(
    df_left=metabooks,
    df_right=amazon_books, 
    candidates=embedding_blocker_m2a,
    comparators=comparators_m2a,
    weights=[0.4, 0.4, 0.2], 
    threshold=0.7,
    id_column='id'
)

# matching metabooks > goodreads
em_corr_m2g = matcher.match(
    df_left=metabooks,
    df_right=goodreads, 
    candidates=embedding_blocker_m2g,
    comparators=comparators_m2g,
    weights=[0.5, 0.25, 0.05, 0.05, 0.05, 0.05, 0.05], 
    threshold=0.7,
    id_column='id'
)

In [ ]:
em_corr_m2a.to_csv(PIPELINE_DIR / "em_corr_m2a.csv", index=False)
em_corr_m2g.to_csv(PIPELINE_DIR / "em_corr_m2g.csv", index=False)

In [ ]:
# data fusion for embedding blocker
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

amazon_books.attrs["trust_score"] = 3
metabooks.attrs["trust_score"] = 2
goodreads.attrs["trust_score"] = 1

all_embedding_correspondences = pd.concat([em_corr_m2a, em_corr_m2g], ignore_index=True)

len(all_embedding_correspondences)

In [ ]:
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title_norm', longest_string)
strategy.add_attribute_fuser('author_norm', longest_string)
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('rating', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres', union)

In [ ]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_embedding_blocker.jsonl")

fused_embedding_blocker = engine.run(
    datasets=[amazon_books, metabooks, goodreads],
    correspondences=all_embedding_correspondences,
    id_column="id",
    include_singletons=False,
)

print(f'Fused rows: {len(fused_embedding_blocker):,}')
display(fused_embedding_blocker.head())